# BM25 vs Embeddings: Why Hybrid Retrieval Wins

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/agents/retrieval_bm25_embeddings_hybrid.ipynb)

Companion notebook to [the post](https://sesen.ai/blog/bm25-vs-embeddings-hybrid-retrieval).

Three retrievers over 5,183 real abstracts with 300 labelled queries:

1. **BM25**, built from the formula with NumPy and scipy.sparse
2. **Dense**, mean-pooled MiniLM embeddings scored by cosine similarity
3. **Hybrid**, reciprocal rank fusion over the two ranked lists

No training anywhere. Runs on CPU; a few minutes end to end, most of it spent
encoding documents.

In [ ]:
!pip install -q transformers torch scipy numpy matplotlib

In [ ]:
import io, json, re, time, urllib.request, zipfile
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import scipy.sparse as sp
import torch
import transformers
from transformers import AutoModel, AutoTokenizer

transformers.logging.set_verbosity_error()   # the long-document warning below is the point

DATA = Path("scifact")
SCIFACT_URL = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip"
ENCODER = "sentence-transformers/all-MiniLM-L6-v2"

## 1. The corpus and its judgements

SciFact, from the BEIR benchmark: scientific abstracts, scientific claims used as
queries, and human relevance judgements linking the two. The judgements are the
point. Without them you cannot tell whether a retrieval change helped.

In [ ]:
def load_scifact(split="test"):
    if not (DATA / "corpus.jsonl").exists():
        with urllib.request.urlopen(SCIFACT_URL, timeout=120) as response:
            blob = response.read()
        with zipfile.ZipFile(io.BytesIO(blob)) as archive:
            archive.extractall(".")

    doc_ids, titles, texts = [], [], []
    for line in open(DATA / "corpus.jsonl", encoding="utf-8"):
        record = json.loads(line)
        doc_ids.append(record["_id"])
        titles.append(record["title"])
        texts.append(f"{record['title']} {record['text']}".strip())

    all_queries = {}
    for line in open(DATA / "queries.jsonl", encoding="utf-8"):
        record = json.loads(line)
        all_queries[record["_id"]] = record["text"]

    qrels = {}
    with open(DATA / "qrels" / f"{split}.tsv", encoding="utf-8") as handle:
        next(handle)
        for line in handle:
            qid, did, score = line.rstrip("\n").split("\t")
            if int(score) > 0:
                qrels.setdefault(qid, set()).add(did)

    index = {d: i for i, d in enumerate(doc_ids)}
    qids = sorted(qrels)
    return {
        "doc_ids": doc_ids,
        "titles": titles,
        "texts": texts,
        "queries": [all_queries[q] for q in qids],
        "gold": [{index[d] for d in qrels[q] if d in index} for q in qids],
    }


corpus = load_scifact()
texts, queries, gold = corpus["texts"], corpus["queries"], corpus["gold"]
lengths = np.array([len(t.split()) for t in texts])
print(f"{len(texts)} documents, {len(queries)} queries")
print(f"mean document length {lengths.mean():.0f} words")
print(f"relevant documents per query: {np.mean([len(g) for g in gold]):.2f}")
print("\nexample query:", queries[0])

## 2. BM25 from the formula

One weight per (term, document) pair, and three factors inside it:

```
score(q, d) = Σ  idf(t) · tf(t,d)·(k1+1) / (tf(t,d) + k1·(1 - b + b·|d|/avgdl))
```

- **idf** rewards rare words. A term in one document out of 5,183 gets the
  maximum weight; a term in half of them gets almost none.
- **saturation** stops the tenth mention of a word from counting ten times as
  much as the first. `k1` sets how fast the curve flattens.
- **length normalisation** discounts long documents, which would otherwise win
  by accumulating matches. `b` sets how hard.

Every weight is computed once and stored term-major. That is an inverted index,
and it makes scoring a walk over one posting list per query term.

In [ ]:
TOKEN = re.compile(r"[a-z0-9]+")

def tokenise(text):
    return TOKEN.findall(text.lower())


class BM25:
    def __init__(self, docs, k1=0.9, b=0.4,
                 use_idf=True, use_saturation=True, use_length_norm=True):
        self.vocab = {}
        rows, cols, vals = [], [], []
        for i, doc in enumerate(docs):
            counts = {}
            for token in tokenise(doc):
                idx = self.vocab.setdefault(token, len(self.vocab))
                counts[idx] = counts.get(idx, 0) + 1
            rows.extend([i] * len(counts))
            cols.extend(counts.keys())
            vals.extend(counts.values())

        n_docs, n_terms = len(docs), len(self.vocab)
        tf = sp.csr_matrix((np.array(vals, dtype=np.float32), (rows, cols)),
                           shape=(n_docs, n_terms))

        doc_len = np.asarray(tf.sum(axis=1)).ravel()
        df = np.asarray((tf > 0).sum(axis=0)).ravel()
        idf = np.log(1.0 + (n_docs - df + 0.5) / (df + 0.5))
        self.idf = idf if use_idf else np.ones_like(idf)
        self.df = df

        norm = (1.0 - b + b * doc_len / doc_len.mean()
                if use_length_norm else np.ones_like(doc_len))

        coo = tf.tocoo()
        raw_tf = coo.data
        if use_saturation:
            weighted = raw_tf * (k1 + 1.0) / (raw_tf + k1 * norm[coo.row])
        else:
            weighted = raw_tf / norm[coo.row]
        data = weighted * self.idf[coo.col]

        postings = sp.csc_matrix((data, (coo.row, coo.col)), shape=(n_docs, n_terms))
        self.n_docs = n_docs
        self.starts = postings.indptr
        self.docs_for_term = postings.indices
        self.weight_for_posting = postings.data

    def score(self, query):
        scores = np.zeros(self.n_docs, dtype=np.float32)
        for token in tokenise(query):
            term = self.vocab.get(token)
            if term is None:
                continue                       # a word nobody wrote cannot match
            span = slice(self.starts[term], self.starts[term + 1])
            scores[self.docs_for_term[span]] += self.weight_for_posting[span]
        return scores

    def rank(self, query, k=100):
        scores = self.score(query)
        k = min(k, len(scores))
        top = np.argpartition(-scores, k - 1)[:k]
        return top[np.argsort(-scores[top])]


started = time.time()
bm25 = BM25(texts)
print(f"indexed {bm25.n_docs} documents in {time.time() - started:.2f}s")
print(f"vocabulary {len(bm25.vocab):,} terms, {len(bm25.weight_for_posting):,} postings")

started = time.time()
lexical = [bm25.rank(q, k=100) for q in queries]
print(f"scored {len(queries)} queries in {time.time() - started:.3f}s")

## 3. The dense index

Mean-pool the token vectors over the attention mask, then L2 normalise so that
cosine similarity is a plain dot product. The whole index becomes one matrix.

Watch the truncation number this cell prints. The encoder's window is 256 tokens
and nothing warns you when a document exceeds it.

In [ ]:
device = "cuda" if torch.cuda.is_available() else (
    "mps" if torch.backends.mps.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(ENCODER)
model = AutoModel.from_pretrained(ENCODER).to(device).eval()
print("device:", device)


def embed(texts, batch=64, max_length=256):
    out = []
    with torch.no_grad():
        for start in range(0, len(texts), batch):
            enc = tokenizer(texts[start:start + batch], padding=True,
                            truncation=True, max_length=max_length,
                            return_tensors="pt").to(device)
            hidden = model(**enc).last_hidden_state
            mask = enc["attention_mask"].unsqueeze(-1).float()
            pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
            out.append(torch.nn.functional.normalize(pooled, dim=-1).cpu().numpy())
    return np.vstack(out).astype(np.float32)


token_lengths = np.array([len(tokenizer.tokenize(t)) + 2 for t in texts])
print(f"documents over the 256-token window: {(token_lengths > 256).mean():.1%}"
      f" (mean {token_lengths.mean():.0f} tokens)")

started = time.time()
doc_vecs = embed(texts)
query_vecs = embed(queries)
print(f"encoded in {time.time() - started:.0f}s, dimension {doc_vecs.shape[1]}")

similarity = query_vecs @ doc_vecs.T
dense = [np.argsort(-row)[:100] for row in similarity]

## 4. Reciprocal rank fusion

A BM25 score of 31.4 and a cosine similarity of 0.71 live on different scales, so
combining them means choosing a normalisation and a weight. Rank fusion throws
the scores away and keeps only the positions:

```
RRF(d) = Σ 1 / (k + rank of d in list r)
```

With `k = 60`, rank 1 contributes 0.0164 and rank 3 contributes 0.0159. One
retriever's confident first pick cannot outvote a document that both put near
the top.

In [ ]:
def rrf(rankings, k=60, top=100):
    scores = {}
    for ranking in rankings:
        for position, doc in enumerate(ranking, start=1):
            scores[int(doc)] = scores.get(int(doc), 0.0) + 1.0 / (k + position)
    order = sorted(scores.items(), key=lambda kv: -kv[1])
    return np.array([doc for doc, _ in order[:top]], dtype=int)


hybrid = [rrf([a, b]) for a, b in zip(lexical, dense)]
print("fusion of the top-10 lists for query 0:", hybrid[0][:10])

## 5. Evaluation

Retrieval has no accuracy. It returns a ranked list, so the metric has to care
about position.

- **recall@k**: are the relevant documents anywhere in the top k? Set k to the
  number of chunks your prompt will hold.
- **MRR@10**: reciprocal rank of the first hit. Use it when one good document is
  enough.
- **nDCG@10**: each hit discounted by log2(rank+1), divided by the best possible
  arrangement. Use it when several documents matter and their order matters.

In [ ]:
def recall_at_k(ranked, gold, k):
    return float(np.mean([len(set(r[:k].tolist()) & g) / len(g)
                          for r, g in zip(ranked, gold)]))


def mrr_at_k(ranked, gold, k=10):
    out = []
    for r, g in zip(ranked, gold):
        hit = [i for i, doc in enumerate(r[:k], start=1) if int(doc) in g]
        out.append(1.0 / hit[0] if hit else 0.0)
    return float(np.mean(out))


def ndcg_at_k(ranked, gold, k=10):
    out = []
    for r, g in zip(ranked, gold):
        dcg = sum((1.0 if int(doc) in g else 0.0) / np.log2(i + 2)
                  for i, doc in enumerate(r[:k]))
        ideal = sum(1.0 / np.log2(i + 2) for i in range(min(len(g), k)))
        out.append(dcg / ideal if ideal else 0.0)
    return float(np.mean(out))


def evaluate(ranked, gold):
    return {
        "nDCG@10": ndcg_at_k(ranked, gold, 10),
        "MRR@10": mrr_at_k(ranked, gold, 10),
        "recall@1": recall_at_k(ranked, gold, 1),
        "recall@10": recall_at_k(ranked, gold, 10),
        "recall@100": recall_at_k(ranked, gold, 100),
    }


runs = {"BM25": lexical, "dense": dense, "hybrid": hybrid}
metrics = list(evaluate(lexical, gold))
print(f"{'':<10}" + "".join(f"{m:>11}" for m in metrics))
for name, ranked in runs.items():
    scores = evaluate(ranked, gold)
    print(f"{name:<10}" + "".join(f"{scores[m]:>11.4f}" for m in metrics))

Expect roughly BM25 0.661, dense 0.645, hybrid 0.694 on nDCG@10. BM25, with no
neural network in it, beats the embedding model on the metric that matters most
for RAG, and fusion beats both on every column at once.

## 6. Where the two retrievers disagree

The aggregate hides what matters. Fusion only pays if the two retrievers fail on
*different* queries, so count that directly.

In [ ]:
def first_hit(ranked, gold):
    hit = np.where(np.isin(ranked, list(gold)))[0]
    return int(hit[0]) + 1 if len(hit) else 10_000


per_query = {name: np.array([ndcg_at_k([r], [g], 10) for r, g in zip(ranked, gold)])
             for name, ranked in runs.items()}
lex_rank = np.array([first_hit(r, g) for r, g in zip(lexical, gold)])
dense_rank = np.array([first_hit(r, g) for r, g in zip(dense, gold)])
delta = per_query["BM25"] - per_query["dense"]

print(f"BM25 strictly better : {(delta > 0).sum()}")
print(f"dense strictly better: {(delta < 0).sum()}")
print(f"tied                 : {(delta == 0).sum()}")
print()
print(f"dense finds it, BM25 misses : {((dense_rank <= 100) & (lex_rank > 100)).sum()}")
print(f"BM25 finds it, dense misses : {((lex_rank <= 100) & (dense_rank > 100)).sum()}")
print(f"neither finds it            : {((lex_rank > 100) & (dense_rank > 100)).sum()}")

for label, order in [("dense wins by the most", np.argsort(delta)),
                     ("BM25 wins by the most", np.argsort(-delta))]:
    print(f"\n{label}:")
    for i in order[:2]:
        doc = sorted(gold[i])[0]
        content = {w for w in tokenise(queries[i]) if len(w) > 4}
        shared = content & set(tokenise(texts[doc]))
        print(f"  query : {queries[i]}")
        print(f"  gold  : {corpus['titles'][doc][:88]}")
        print(f"  BM25 rank {lex_rank[i]}, dense rank {dense_rank[i]},"
              f" shared content words {len(shared)}/{len(content)}")

Two named problems produce those two tails.

**The vocabulary gap.** Query and document say the same thing in different words.
BM25 sums weights over shared terms, so with nothing shared it scores near zero.
The embedding connects the phrasings because its training data did.

**The lexical gap.** The query turns on a rare string the encoder cannot
represent: a gene symbol, a part number, an error code. BM25 gives it a huge IDF
and wins; the wordpiece tokeniser shreds it into fragments and dense loses.

Each gap has its own fix and neither fix helps the other. Running both gets both.

## 7. Isolating the lexical gap

Build queries where a rare identifier is the only thing distinguishing the
target: one token that occurs in exactly one document, padded with three words
common enough to name a topic and pin nothing.

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

STOPWORDS = frozenset(ENGLISH_STOP_WORDS)
IDENTIFIER = re.compile(r"^[a-z][a-z0-9]{3,11}$")
CONTEXT_DF = (100, 1000)


def rare_identifier_queries(n=40, seed=0):
    postings = {}
    for i, text in enumerate(texts):
        for token in set(tokenise(text)):
            postings.setdefault(token, []).append(i)
    df = {t: len(d) for t, d in postings.items()}

    candidates = []
    for token, docs in sorted(postings.items()):
        if len(docs) != 1 or not IDENTIFIER.match(token):
            continue
        letters = sum(c.isalpha() for c in token)
        if letters < 2 or letters == len(token):
            continue
        doc, seen, context = docs[0], set(), []
        for word in tokenise(texts[doc]):
            if word in seen or word in STOPWORDS or len(word) < 4:
                continue
            if CONTEXT_DF[0] <= df.get(word, 0) <= CONTEXT_DF[1]:
                seen.add(word)
                context.append(word)
        if len(context) >= 3:
            candidates.append((token, doc, context[:3]))

    rng = np.random.default_rng(seed)
    picked = rng.choice(len(candidates), size=min(n, len(candidates)), replace=False)
    return [(f"{candidates[i][0]} {' '.join(candidates[i][2])}",
             " ".join(candidates[i][2]), candidates[i][1]) for i in sorted(picked)]


adversarial = rare_identifier_queries()
adv_texts = [q for q, _, _ in adversarial]
adv_gold = [{d} for _, _, d in adversarial]
print("examples:", adv_texts[:3])
print("how the tokeniser sees one identifier:",
      tokenizer.tokenize(adv_texts[0].split()[0]))

adv_vecs = embed(adv_texts)
adv_lex = [bm25.rank(q, k=100) for q in adv_texts]
adv_dense = [np.argsort(-row)[:100] for row in (adv_vecs @ doc_vecs.T)]
adv_hybrid = [rrf([a, b]) for a, b in zip(adv_lex, adv_dense)]

print(f"\n{'':<10}" + "".join(f"{m:>11}" for m in metrics) + "   rank-1   missed")
for name, ranked in [("BM25", adv_lex), ("dense", adv_dense), ("hybrid", adv_hybrid)]:
    scores = evaluate(ranked, adv_gold)
    ranks = np.array([first_hit(r, g) for r, g in zip(ranked, adv_gold)])
    print(f"{name:<10}" + "".join(f"{scores[m]:>11.4f}" for m in metrics)
          + f"   {(ranks == 1).sum():>6}   {(ranks > 100).sum():>6}")

control = embed([c for _, c, _ in adversarial])
ctl_dense = [np.argsort(-row)[:100] for row in (control @ doc_vecs.T)]
print("\nsame queries with the identifier deleted:")
print(f"  BM25  nDCG@10 {ndcg_at_k([bm25.rank(c, k=100) for _, c, _ in adversarial], adv_gold):.4f}")
print(f"  dense nDCG@10 {ndcg_at_k(ctl_dense, adv_gold):.4f}")

BM25 puts the right document first on all forty. The dense index manages about a
third and loses a fifth of them past rank 100, which for a pipeline pulling five
chunks means the answer never reaches the model.

Note what fusion costs here: hybrid scores well below BM25 alone, because half
the votes come from a retriever that is wrong. Fusion is insurance, and insurance
has a premium.

## 8. What each part of BM25 is doing

Switch off one factor at a time and re-measure.

In [ ]:
for name, kwargs in [("full BM25", {}),
                     ("no saturation", {"use_saturation": False}),
                     ("no IDF", {"use_idf": False}),
                     ("no length norm", {"use_length_norm": False})]:
    variant = BM25(texts, **kwargs)
    score = ndcg_at_k([variant.rank(q, k=100) for q in queries], gold, 10)
    print(f"{name:<16} nDCG@10 {score:.4f}")

print("\nk1 sweep (saturation speed):")
for k1 in (0.0, 0.3, 0.6, 0.9, 1.2, 2.0, 3.0):
    variant = BM25(texts, k1=k1)
    print(f"  k1={k1:<4} {ndcg_at_k([variant.rank(q, k=100) for q in queries], gold, 10):.4f}")

print("\nb sweep (length normalisation):")
for b in (0.0, 0.4, 0.8, 1.0):
    variant = BM25(texts, b=b)
    print(f"  b={b:<4} {ndcg_at_k([variant.rank(q, k=100) for q in queries], gold, 10):.4f}")

IDF and saturation each carry around twelve points of nDCG. Length normalisation
carries about one, because every document here is an abstract; on a corpus mixing
tweets with legal filings that number would look very different.

At `k1 = 0` the term count is discarded entirely and every present term
contributes its IDF. That still scores about 0.614, so counts are worth roughly
five points on top of rarity, not fifty.

## 9. Rank fusion against score fusion

The alternative to RRF normalises both score distributions and blends them with a
weight. Sweep both and compare the shapes, not just the peaks.

In [ ]:
lex_scores = [bm25.score(q) for q in queries]

print("RRF constant k:")
for k in (1, 5, 10, 20, 60, 120, 300):
    fused = [rrf([a, b], k=k) for a, b in zip(lexical, dense)]
    print(f"  k={k:<5} {ndcg_at_k(fused, gold, 10):.4f}")

print("\nscore fusion weight alpha (1.0 = BM25 only, 0.0 = dense only):")
for alpha in (0.0, 0.2, 0.4, 0.6, 0.8, 1.0):
    fused = []
    for ls, ds in zip(lex_scores, similarity):
        a = (ls - ls.min()) / (ls.max() - ls.min() + 1e-9)
        b_ = (ds - ds.min()) / (ds.max() - ds.min() + 1e-9)
        fused.append(np.argsort(-(alpha * a + (1 - alpha) * b_))[:100])
    print(f"  alpha={alpha:<4} {ndcg_at_k(fused, gold, 10):.4f}")

Tuned score fusion wins on the peak, by about two points. Look at the shapes
though: every setting of the RRF constant beats both individual retrievers, so
the worst an untuned rank fusion can do is win by less. Score fusion at the wrong
weight falls below rank fusion, and at the endpoints collapses into a single
retriever.

With a judged query set, tune the weight. Without one, take rank fusion.

## 10. Chunk size

Usually copied from a tutorial and never revisited. Index the same corpus at
several granularities and evaluate at document level so the numbers stay
comparable. (The post also runs 32-word chunks; they double the encoding time
for one more point on the curve.)

In [ ]:
def chunk_corpus(texts, size):
    chunks, owner = [], []
    for i, text in enumerate(texts):
        words = text.split()
        if len(words) <= size:
            chunks.append(text)
            owner.append(i)
            continue
        for start in range(0, len(words), size):
            window = words[start:start + size]
            if len(window) < size // 2 and start > 0:
                break
            chunks.append(" ".join(window))
            owner.append(i)
    return chunks, np.array(owner)


def chunks_to_docs(ranked_chunks, owner, k):
    seen, out = set(), []
    for chunk in ranked_chunks:
        doc = int(owner[int(chunk)])
        if doc not in seen:
            seen.add(doc)
            out.append(doc)
        if len(out) >= k:
            break
    return np.array(out, dtype=int)


print(f"{'chunk':<8}{'vectors':>9}{'BM25':>9}{'dense':>9}{'hybrid':>9}")
for size in (64, 128):
    chunks, owner = chunk_corpus(texts, size)
    c_bm25 = BM25(chunks)
    c_vecs = embed(chunks)
    c_lex = [chunks_to_docs(c_bm25.rank(q, k=300), owner, 100) for q in queries]
    c_dense = [chunks_to_docs(np.argsort(-row)[:300], owner, 100)
               for row in (query_vecs @ c_vecs.T)]
    c_hybrid = [rrf([a, b]) for a, b in zip(c_lex, c_dense)]
    print(f"{size:<8}{len(chunks):>9,}"
          f"{ndcg_at_k(c_lex, gold, 10):>9.4f}"
          f"{ndcg_at_k(c_dense, gold, 10):>9.4f}"
          f"{ndcg_at_k(c_hybrid, gold, 10):>9.4f}")

print(f"{'whole':<8}{len(texts):>9,}"
      f"{ndcg_at_k(lexical, gold, 10):>9.4f}"
      f"{ndcg_at_k(dense, gold, 10):>9.4f}"
      f"{ndcg_at_k(hybrid, gold, 10):>9.4f}")

The two curves run in opposite directions. Dense retrieval improves as chunks
shrink, because one 384-dimensional vector cannot hold a 215-word abstract and
because chunking sidesteps the 256-token truncation. BM25 degrades, because
splitting a document destroys the term co-occurrence that made it findable.

The chunk size that suits your embeddings is close to the worst one for your
lexical index. If you can afford two indexes, chunk them differently.

## Exercises

1. **Stem the tokeniser.** Add a Porter stemmer to `tokenise` so that
   "inhibits" and "inhibition" collide. Does BM25 improve, and does it lose any
   of the identifier queries?
2. **Swap the encoder.** Replace MiniLM with `BAAI/bge-small-en-v1.5` or
   `intfloat/e5-small-v2` (both need their own query prefix). How much of the
   gap to BM25 closes, and does the identifier result move at all?
3. **Fuse three lists.** Add a title-only BM25 index as a third retriever and
   fuse all three. Rank fusion takes any number of lists without new weights.
4. **Find the ceiling.** For each query, take the better of the two ranks and
   compute nDCG@10 on that oracle. That is what a perfect router would score,
   and the distance from hybrid to it is what fusion leaves on the table.
5. **Break the identifier test.** Change `CONTEXT_DF` so the padding words are
   rare rather than common. Does BM25 stay perfect, and does dense recover?